# Neural style transfer

_Deep Learning_

**Keep a photo's content but repaint it in another image's artistic style.**

Neural style transfer mixes two images: the content of a photo and the style of a painting.

     The result keeps what is in your photo but paints it in the brushstrokes and colors of the artwork.

---

This notebook is a practice scaffold for the **Neural style transfer** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn.functional as F

def gram(feat):
    # feat: (batch, channels, h, w) -> channel correlation matrix
    b, c, h, w = feat.shape
    f = feat.view(c, h * w)
    return (f @ f.t()) / (c * h * w)   # the "style" of a layer

content_feat = torch.randn(1, 16, 8, 8)        # features of the content photo
style_feat = torch.randn(1, 16, 8, 8)          # features of the style painting
img = torch.randn(1, 16, 8, 8, requires_grad=True)   # the image we optimize
opt = torch.optim.Adam([img], lr=0.05)

for step in range(5):
    opt.zero_grad()
    content_loss = F.mse_loss(img, content_feat)             # keep the content
    style_loss = F.mse_loss(gram(img), gram(style_feat))     # match the style
    (content_loss + 1e3 * style_loss).backward()             # blend the two
    opt.step()
print("final style loss:", round(F.mse_loss(gram(img), gram(style_feat)).item(), 5))

## Visualize the data & results

_As we optimize a blank image toward a real digit, does the content loss fall to zero?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
target = digits.images[8].astype(float)  # real 8x8 image of an 8
cur = np.zeros_like(target)              # start from a blank canvas

losses = []
for step in range(16):
    losses.append(((cur - target) ** 2).mean())   # content loss
    cur += 0.4 * (target - cur)                    # gradient step on pixels

plt.plot(losses, color="#c89bff", label="content loss")
plt.title("Content loss while optimizing an image toward a real digit")
plt.xlabel("step"); plt.ylabel("mean-squared loss")
plt.legend()
plt.show()